<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet6%20(ANN)/Praktikum%203%20(JS6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=7669b00b9f29714fb1e175da357a868fed840ff2d98ae7c7e57d5318a1328e97
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [3]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[303 574 467 592 787]]
Distances: [[0.0135662  0.01637629 0.01714669 0.02629166 0.02933748]]
Waktu: 0.01286172866821289 detik

=== HNSW ===
Indices: [[303 574 467 592 787]]
Distances: [[0.00018404 0.00026818 0.00029401 0.00069125 0.00086069]]
Waktu: 0.0002799034118652344 detik


Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1jt data, 2D vs 5D data. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

In [4]:
import hnswlib
import numpy as np
import time
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Konfigurasi pengujian variasi
experiments = [
    {"metric": "Euclidean (L2)", "hnsw_space": "l2", "dim": 2, "n_data": 1000},
    {"metric": "Euclidean (L2)", "hnsw_space": "l2", "dim": 2, "n_data": 1000000},
    {"metric": "Euclidean (L2)", "hnsw_space": "l2", "dim": 5, "n_data": 1000},
    {"metric": "Euclidean (L2)", "hnsw_space": "l2", "dim": 5, "n_data": 1000000},
    {"metric": "Cosine", "hnsw_space": "cosine", "dim": 2, "n_data": 1000},
    {"metric": "Cosine", "hnsw_space": "cosine", "dim": 2, "n_data": 1000000},
    {"metric": "Cosine", "hnsw_space": "cosine", "dim": 5, "n_data": 1000},
    {"metric": "Cosine", "hnsw_space": "cosine", "dim": 5, "n_data": 1000000},
]

k = 5
results = []

for exp in experiments:
    np.random.seed(42)
    dim = exp["dim"]
    n_data = exp["n_data"]

    # Sintesis data dan query
    data = np.random.random((n_data, dim)).astype(np.float32)
    query = np.array([[0.5] * dim], dtype=np.float32)

    # --- 1. Exact NN ---
    metric_sklearn = 'euclidean' if exp["metric"] == "Euclidean (L2)" else 'cosine'
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric_sklearn)
    nn.fit(data)

    start = time.time()
    _, indices_exact = nn.kneighbors(query)
    t_exact = (time.time() - start) * 1000  # ms

    # --- 2. HNSW ---
    p = hnswlib.Index(space=exp["hnsw_space"], dim=dim)
    p.init_index(max_elements=n_data, ef_construction=100, M=16)
    p.add_items(data)
    p.set_ef(50)

    start = time.time()
    labels_hnsw, _ = p.knn_query(query, k=k)
    t_hnsw = (time.time() - start) * 1000  # ms

    # Simpan hasil
    results.append({
        "Distance Metric": exp["metric"],
        "Dimensi": f"{dim}D",
        "Jumlah Data": f"{n_data:,}",
        "Hasil Index (Exact vs HNSW)": f"{indices_exact[0].tolist()} vs {labels_hnsw[0].tolist()}",
        "Waktu Komputasi (Exact vs HNSW)": f"{t_exact:.4f} ms vs {t_hnsw:.4f} ms"
    })

df_hnsw = pd.DataFrame(results)
display(df_hnsw)

,Distance Metric,Dimensi,Jumlah Data,Hasil Index (Exact vs HNSW),Waktu Komputasi (Exact vs HNSW)
0,Euclidean (L2),2D,"1,000","[137, 170, 750, 316, 991] vs [137, 170, 750, 3...",10.6657 ms vs 0.1345 ms
1,Euclidean (L2),2D,"1,000,000","[286942, 583487, 695123, 948064, 829288] vs [2...",59.6111 ms vs 0.1695 ms
2,Euclidean (L2),5D,"1,000","[895, 451, 375, 386, 411] vs [895, 451, 375, 3...",0.9539 ms vs 0.1338 ms
3,Euclidean (L2),5D,"1,000,000","[794791, 501340, 912100, 84056, 660781] vs [79...",34.5235 ms vs 0.2909 ms
4,Cosine,2D,"1,000","[2, 135, 413, 210, 700] vs [2, 135, 413, 210, ...",14.6914 ms vs 0.0901 ms
5,Cosine,2D,"1,000,000","[469917, 383546, 49557, 317128, 87009] vs [870...",36.9320 ms vs 0.1788 ms
6,Cosine,5D,"1,000","[248, 386, 229, 230, 437] vs [248, 386, 229, 2...",1.9484 ms vs 0.0949 ms
7,Cosine,5D,"1,000,000","[422252, 464287, 605535, 319582, 869360] vs [4...",75.6695 ms vs 0.1979 ms
